In [1]:
from pathlib import Path
from pyspark.sql import SparkSession

# Emplacement des données du TP3
DATA_ROOT = Path.home() / "retail-intelligence" / "tp3" / "data"

# Création de la session Spark
spark = (
    SparkSession.builder
    .appName("RetailIntelligence-TP4-Inventaire")
    .master("local[*]").config("spark.hadoop.fs.defaultFS", "file:///").getOrCreate()
)

print("Spark version :", spark.version)
print("Data root :", DATA_ROOT)
print("Data root exists :", DATA_ROOT.exists())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/20 22:27:02 WARN Utils: Your hostname, NAVID-DELL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/20 22:27:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/navid/retail-intelligence/tp4/retail_intel_tp4_venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/20 22:27:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.3.0
Data root : /home/navid/retail-intelligence/tp3/data
Data root exists : True


In [2]:
def inspect_dataset(dataset_name, dataset_path):
    path = Path(dataset_path)

    print("=" * 60)
    print("DATASET :", dataset_name)
    print("PATH :", path)

    if not path.exists():
        print("ERREUR : chemin introuvable")
        return

    parquet_files = list(path.rglob("*.parquet"))

    if not parquet_files:
        print("Aucun fichier Parquet trouvé")
        return

    df = spark.read.parquet(str(path))

    print("\nSCHÉMA :")
    df.printSchema()

    print("\nNOMBRE DE COLONNES :", len(df.columns))

    print("\nCOLONNES :")
    print(df.columns)

    print("\nEXEMPLES DE DONNÉES :")
    df.show(5, truncate=False)

    print("\nNOMBRE DE LIGNES :")
    print(df.count())

    print("=" * 60)

In [3]:
silver_transformed = DATA_ROOT / "silver-transformed"
gold = DATA_ROOT / "gold"

datasets = {
    "sales_by_store": silver_transformed / "sales_by_store",
    "sales_by_store_date": silver_transformed / "sales_by_store_date",
    "sales_by_item_date": silver_transformed / "sales_by_item_date",
    "inventory_by_item_date": silver_transformed / "inventory_by_item_date",
    "sales_inventory_by_item_date": silver_transformed / "sales_inventory_by_item_date",
    "sales_performance_by_store": gold / "sales_performance_by_store"
}

for name, path in datasets.items():
    inspect_dataset(name, path)

DATASET : sales_by_store
PATH : /home/navid/retail-intelligence/tp3/data/silver-transformed/sales_by_store



SCHÉMA :
root
 |-- store_sk: long (nullable = true)
 |-- total_revenue_inc_tax: decimal(17,2) (nullable = true)
 |-- total_net_paid: decimal(17,2) (nullable = true)
 |-- total_net_profit: decimal(17,2) (nullable = true)
 |-- total_quantity_sold: long (nullable = true)
 |-- number_of_sales_lines: long (nullable = true)
 |-- number_of_tickets: long (nullable = true)
 |-- _grain: string (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_original_file: string (nullable = true)
 |-- _processing_stage: string (nullable = true)


NOMBRE DE COLONNES : 11

COLONNES :
['store_sk', 'total_revenue_inc_tax', 'total_net_paid', 'total_net_profit', 'total_quantity_sold', 'number_of_sales_lines', 'number_of_tickets', '_grain', '_source_dataset', '_source_original_file', '_processing_stage']

EXEMPLES DE DONNÉES :


+--------+---------------------+--------------+----------------+-------------------+---------------------+-----------------+----------+---------------+---------------------------------------------------------------------------+---------------------------------+
|store_sk|total_revenue_inc_tax|total_net_paid|total_net_profit|total_quantity_sold|number_of_sales_lines|number_of_tickets|_grain    |_source_dataset|_source_original_file                                                      |_processing_stage                |
+--------+---------------------+--------------+----------------+-------------------+---------------------+-----------------+----------+---------------+---------------------------------------------------------------------------+---------------------------------+
|9       |101109360.86         |96748356.68   |-46015802.68    |2830684            |56040                |7435             |un magasin|store_sales    |/home/navid/retail-intelligence/tp3/data/bronze/store_sales/sto

In [4]:
df = spark.read.parquet(
    str(silver_transformed / "sales_by_store_date")
)

df.printSchema()

root
 |-- store_sk: long (nullable = true)
 |-- date_sk: long (nullable = true)
 |-- total_revenue_inc_tax: decimal(17,2) (nullable = true)
 |-- total_net_paid: decimal(17,2) (nullable = true)
 |-- total_net_profit: decimal(17,2) (nullable = true)
 |-- total_quantity_sold: long (nullable = true)
 |-- number_of_sales_lines: long (nullable = true)
 |-- number_of_tickets: long (nullable = true)
 |-- _grain: string (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_original_file: string (nullable = true)
 |-- _processing_stage: string (nullable = true)



In [5]:
from pyspark.sql import functions as F

duplicates = (
    df.groupBy("store_sk", "date_sk")
      .count()
      .filter(F.col("count") > 1)
)

print("Nombre de combinaisons dupliquées :",
      duplicates.count())

[Stage 31:>                                                         (0 + 1) / 1]

Nombre de combinaisons dupliquées : 0


In [6]:
import pandas as pd

quality_path = (
    DATA_ROOT.parent
    / "reports"
    / "quality"
    / "quality_report_gold.csv"
)

if quality_path.exists(): 
    quality_report = pd.read_csv(quality_path, sep=";", encoding="utf-8-sig")

    display(quality_report.head(50))

    print("Colonnes du rapport :")
    print(quality_report.columns.tolist())
else:
    print("Rapport qualité introuvable")

,controle,regle,resultat,statut,details
0,Volume,Le nombre de lignes doit être égal à 12,12,PASS,Une ligne est attendue par magasin.
1,Structure,Toutes les colonnes indispensables doivent êtr...,Aucune colonne manquante,PASS,NaN
2,Complétude,store_sk ne doit pas contenir de valeur nulle,0 valeur(s) nulle(s),PASS,NaN
3,Complétude,revenue_rank ne doit pas contenir de valeur nulle,0 valeur(s) nulle(s),PASS,NaN
4,Complétude,is_best_revenue_store ne doit pas contenir de ...,0 valeur(s) nulle(s),PASS,NaN
5,Complétude,total_revenue_inc_tax ne doit pas contenir de ...,0 valeur(s) nulle(s),PASS,NaN
6,Complétude,global_revenue_inc_tax ne doit pas contenir de...,0 valeur(s) nulle(s),PASS,NaN
7,Complétude,revenue_share_pct ne doit pas contenir de vale...,0 valeur(s) nulle(s),PASS,NaN
8,Complétude,total_net_paid ne doit pas contenir de valeur ...,0 valeur(s) nulle(s),PASS,NaN
9,Complétude,total_net_profit ne doit pas contenir de valeu...,0 valeur(s) nulle(s),PASS,NaN


Colonnes du rapport :
['controle', 'regle', 'resultat', 'statut', 'details']


In [7]:
from pyspark.sql import functions as F

df_cross = spark.read.parquet(
    str(silver_transformed / "sales_inventory_by_item_date")
)

df_cross.groupBy(
    "flag_inventory_not_matched"
).count().show()

df_cross.select(
    F.count("*").alias("total_lignes"),
    F.sum(
        F.when(
            F.col("total_quantity_on_hand").isNull(), 1
        ).otherwise(0)
    ).alias("stock_null"),
    F.sum(
        F.when(
            F.col("flag_inventory_not_matched") == True, 1
        ).otherwise(0)
    ).alias("non_correspondances")
).show()

+--------------------------+------+
|flag_inventory_not_matched| count|
+--------------------------+------+
|                      true|484701|
|                     false| 80727|
+--------------------------+------+

+------------+----------+-------------------+
|total_lignes|stock_null|non_correspondances|
+------------+----------+-------------------+
|      565428|    484701|             484701|
+------------+----------+-------------------+



In [8]:
datasets_silver = {
    "store_sales": DATA_ROOT / "silver" / "store_sales",
    "inventory": DATA_ROOT / "silver" / "inventory"
}

for name, path in datasets_silver.items():
    inspect_dataset(name, path)

DATASET : store_sales
PATH : /home/navid/retail-intelligence/tp3/data/silver/store_sales

SCHÉMA :
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = t

26/11/20 22:27:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+--------------------+----------------------------+--------------------+----------------------------+---------------+-----------------------+-------------------+---------------------------+----------------+------------------------+----------------+------------------------+---------------+-----------------------+----------------+------------------------+----------------+------------------------+---------------------+-----------------------------+----------------+------------------------+----------------------+------------------------------+------------------+--------------------------+-------------------+---------------------------+------------------

+-----------+-----------+----------------+--------------------+----------------+------------------------+----------------+------------------------+---------------------+-----------------------------+-------------------------+---------------------------------+---------------------+-----------------+----------------------+-------------------------+------------------------+
|inv_date_sk|inv_item_sk|inv_warehouse_sk|inv_quantity_on_hand|_raw_inv_date_sk|invalid_cast_inv_date_sk|_raw_inv_item_sk|invalid_cast_inv_item_sk|_raw_inv_warehouse_sk|invalid_cast_inv_warehouse_sk|_raw_inv_quantity_on_hand|invalid_cast_inv_quantity_on_hand|flag_missing_required|flag_invalid_cast|flag_quantity_negative|duplicate_candidate_count|flag_duplicate_candidate|
+-----------+-----------+----------------+--------------------+----------------+------------------------+----------------+------------------------+---------------------+-----------------------------+-------------------------+---------------------------

In [9]:
spark.stop()
print("Session Spark fermée.")

Session Spark fermée.
